In [2]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[2]))

In [3]:
from __future__ import annotations
import os
import json
import operator
import builtins
from typing import Annotated, Sequence, TypedDict, Optional

In [4]:
from langchain_openai.chat_models import ChatOpenAI
from langchain.schema import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.messages.tool import ToolMessage
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.tools import tool
from langgraph.graph import StateGraph, END

In [5]:
from observability.loki_logger import log_to_loki

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
AZURE_OPENAI_API_KEY=os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_ENDPOINT=os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_VERSION=os.environ["AZURE_OPENAI_API_VERSION"]
TRACELOOP_API_KEY=os.getenv("TRACELOOP_API_KEY")

In [8]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource

resource = Resource.create({"service.name": "financial_agent"})
provider = TracerProvider(resource=resource)

otlp_exporter = OTLPSpanExporter(endpoint="localhost:4317", insecure=True)
provider.add_span_processor(BatchSpanProcessor(otlp_exporter))

trace.set_tracer_provider(provider)
tracer = trace.get_tracer(__name__)

In [ ]:
@tool
def investigate_transaction(account_id: str = None, customer_id: str = None, alert_type: str = None, **kwargs) -> str:
    """Investigate suspicious transactions, fraud alerts, or security concerns."""
    with tracer.start_as_current_span("investigate_transaction"):

        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] investigate_transaction(account_id={account_id}, alert_type={alert_type}, kwargs={kwargs})")
        log_to_loki("tool.investigate_transaction", trace_id, f"account_id={account_id}, alert_type={alert_type}")
        return "investigation_initiated"

@tool
def freeze_account(account_id: str, reason: str, freeze_type: str = "immediate", customer_request: bool = True) -> str:
    """Freeze account to prevent unauthorized access or transactions."""
    with tracer.start_as_current_span("freeze_account"):

        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] freeze_account(account_id={account_id}, reason={reason}, freeze_type={freeze_type})")
        log_to_loki("tool.freeze_account", trace_id, f"account_id={account_id}, reason={reason}")
        return "account_frozen"

@tool
def process_loan_application(customer_id: str, loan_type: str, loan_amount: str = None, **kwargs) -> str:
    """Process loan applications including personal, business, mortgage, and auto loans."""
    with tracer.start_as_current_span("process_loan_application"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] process_loan_application(customer_id={customer_id}, loan_type={loan_type}, amount={loan_amount})")
        log_to_loki("tool.process_loan_application", trace_id, f"customer_id={customer_id}, loan_type={loan_type}")
        return "application_submitted"

@tool
def resolve_dispute(account_id: str = None, customer_id: str = None, dispute_type: str = None, **kwargs) -> str:
    """Handle disputes including unauthorized charges, fees, and credit report errors."""
    with tracer.start_as_current_span("resolve_dispute"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] resolve_dispute(account_id={account_id}, dispute_type={dispute_type}, kwargs={kwargs})")
        log_to_loki("tool.resolve_dispute", trace_id, f"account_id={account_id}, dispute_type={dispute_type}")
        return "dispute_filed"

@tool
def rebalance_portfolio(customer_id: str, **kwargs) -> str:
    """Manage investment portfolios, retirement planning, and asset allocation."""
    with tracer.start_as_current_span("rebalance_portfolio"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] rebalance_portfolio(customer_id={customer_id}, kwargs={kwargs})")
        log_to_loki("tool.rebalance_portfolio", trace_id, f"customer_id={customer_id}")
        return "portfolio_updated"

@tool
def increase_credit_limit(account_id: str, current_limit: str, requested_limit: str, **kwargs) -> str:
    """Process credit limit increase requests."""
    with tracer.start_as_current_span("increase_credit_limit"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] increase_credit_limit(account_id={account_id}, current={current_limit}, requested={requested_limit})")
        log_to_loki("tool.increase_credit_limit", trace_id, f"account_id={account_id}, increase_request={requested_limit}")
        return "credit_limit_updated"

@tool
def verify_documents(customer_id: str, **kwargs) -> str:
    """Verify customer documents for various banking services."""
    with tracer.start_as_current_span("verify_documents"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] verify_documents(customer_id={customer_id}, kwargs={kwargs})")
        log_to_loki("tool.verify_documents", trace_id, f"customer_id={customer_id}")
        return "documents_verified"

@tool
def update_account(account_id: str = None, customer_id: str = None, **kwargs) -> str:
    """Update account information, add joint holders, close accounts, etc."""
    with tracer.start_as_current_span("update_account"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] update_account(account_id={account_id}, customer_id={customer_id}, kwargs={kwargs})")
        log_to_loki("tool.update_account", trace_id, f"account_id={account_id}, customer_id={customer_id}")
        return "account_updated"

@tool
def process_transaction(customer_id: str, transaction_type: str, **kwargs) -> str:
    """Process various transactions like currency exchange, transfers, etc."""
    with tracer.start_as_current_span("process_transaction"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] process_transaction(customer_id={customer_id}, type={transaction_type}, kwargs={kwargs})")
        log_to_loki("tool.process_transaction", trace_id, f"customer_id={customer_id}, type={transaction_type}")
        return "transaction_processed"

@tool
def send_customer_response(customer_id: str, message: str) -> str:
    """Send a response message to the customer."""
    with tracer.start_as_current_span("send_customer_response"):
        span = trace.get_current_span()
        trace_id = format(span.get_span_context().trace_id, "032x")

        print(f"[TOOL] send_customer_response → {message}")
        log_to_loki("tool.send_customer_response", trace_id, f"customer_id={customer_id}, message={message}")
        return "message_sent"

TOOLS = [
    investigate_transaction, freeze_account, process_loan_application, resolve_dispute,
    rebalance_portfolio, increase_credit_limit, verify_documents, update_account,
    process_transaction, send_customer_response
]